# MF_Gates_Plus_FuzzyOutput using DOG\nDataset: Lake Erie

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.pyplot as plt


In [2]:
def dog(x, mu1=-1.0, mu2=1.0, sigma1=0.5, sigma2=0.5):
    return tf.exp(-tf.square(x - mu1)/ (2 * sigma1**2)) - tf.exp(-tf.square(x - mu2)/ (2 * sigma2**2))

def signed_gaussian(x, sigma=0.5):
    return x * tf.exp(-tf.square(x)/ (2 * sigma**2))

def generalized_bell(x, a=1.0, b=2.0, c_val=0.0):
    return 1.0 / (1.0 + tf.pow(tf.abs((x - c_val) / a), 2*b))

@tf.keras.utils.register_keras_serializable()
class MembershipLSTMSNPCell(layers.Layer):
    def __init__(self, units, mf_type='dog', **kwargs):
        super().__init__(**kwargs)
        self.units, self.mf_type = units, mf_type
        self.state_size, self.output_size = (units, units, units), units
        self.mf = dog if mf_type == 'dog' else signed_gaussian if mf_type == 'sg' else generalized_bell

    def build(self, input_shape):
        input_dim = input_shape[-1]
        self.kernel = self.add_weight(shape=(input_dim, self.units * 4), initializer='glorot_uniform', name='kernel')
        self.recurrent_kernel = self.add_weight(shape=(self.units, self.units * 4), initializer='orthogonal', name='recurrent_kernel')
        self.bias = self.add_weight(shape=(self.units * 4,), initializer='zeros', name='bias')

    def call(self, inputs, states):
        u_tm1 = states[0]  
        z = tf.matmul(inputs, self.kernel) + tf.matmul(u_tm1, self.recurrent_kernel) + self.bias
        z0, z1, z2, z3 = z[:, :self.units], z[:, self.units:2*self.units], z[:, 2*self.units:3*self.units], z[:, 3*self.units:]

        r = tf.tanh(z0)
        c = tf.clip_by_value(self.mf(z1), -1.0, 1.0)
        o = tf.clip_by_value(self.mf(z2), -1.0, 1.0)
        a = tf.tanh(z3)

        u = r * u_tm1 - c * a
        h = o * a
        return h, [u, c, o]

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'mf_type': self.mf_type})
        return config


In [3]:
@tf.keras.utils.register_keras_serializable()
class FuzzyOutputLayer(layers.Layer):
    def __init__(self, units_in, **kwargs):
        super().__init__(**kwargs)
        self.units_in, self.sigma = units_in, 0.5
    def build(self, input_shape):
        self.rule_a = self.add_weight(shape=(4,), initializer='glorot_uniform', name='rule_a')
        self.rule_b = self.add_weight(shape=(4,), initializer='glorot_uniform', name='rule_b')
        self.rule_c = self.add_weight(shape=(4,), initializer='zeros', name='rule_c')
    def call(self, inputs):
        half = self.units_in // 2
        s1 = tf.reduce_mean(inputs[:, :half], axis=-1, keepdims=True)
        s2 = tf.reduce_mean(inputs[:, half:], axis=-1, keepdims=True)
        
        m_l1, m_h1 = tf.exp(-tf.square(s1 - (-1.0))/ (2.0*0.25)), tf.exp(-tf.square(s1 - 1.0)/ (2.0*0.25))
        m_l2, m_h2 = tf.exp(-tf.square(s2 - (-1.0))/ (2.0*0.25)), tf.exp(-tf.square(s2 - 1.0)/ (2.0*0.25))
        w1, w2, w3, w4 = m_l1*m_l2, m_l1*m_h2, m_h1*m_l2, m_h1*m_h2
        
        y1, y2 = self.rule_a[0]*s1 + self.rule_b[0]*s2 + self.rule_c[0], self.rule_a[1]*s1 + self.rule_b[1]*s2 + self.rule_c[1]
        y3, y4 = self.rule_a[2]*s1 + self.rule_b[2]*s2 + self.rule_c[2], self.rule_a[3]*s1 + self.rule_b[3]*s2 + self.rule_c[3]
        return (w1*y1 + w2*y2 + w3*y3 + w4*y4) / (w1 + w2 + w3 + w4 + 1e-8)
    def get_config(self):
        config = super().get_config()
        config.update({'units_in': self.units_in})
        return config


In [4]:
def build_model(input_dim, units, batch_size):
    cell = MembershipLSTMSNPCell(units, mf_type='dog')
    rnn = layers.RNN(cell, return_sequences=False, return_state=True, stateful=True)

    inputs = tf.keras.Input(batch_shape=(batch_size, 1, input_dim))
    x, u_out, c_out, o_out = rnn(inputs)
    outputs = FuzzyOutputLayer(units_in=units)(x)

    model = tf.keras.Model(inputs=inputs, outputs=[outputs, c_out, o_out])
    model.compile(optimizer=tf.keras.optimizers.Adam(clipnorm=1.0), loss=['mean_squared_error', None, None])
    return model


In [5]:
series = pd.read_csv('content/monthly-lake-erie-levels-1921-19.csv', header=0, parse_dates=[0], index_col=0)
raw_values = series.values.flatten()

In [6]:
def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

diff_values = difference(raw_values, 1)
def timeseries_to_supervised(data, lag):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values


In [7]:

supervised = timeseries_to_supervised(diff_values, 1)
train, test = supervised[:-60], supervised[-60:]
scaler = MinMaxScaler(feature_range=(-1, 1))
train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

X_train_raw, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
X_test_raw, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]





X_train = X_train_raw.reshape((X_train_raw.shape[0], 1, 1))
X_test = X_test_raw.reshape((X_test_raw.shape[0], 1, 1))

all_rmse, all_mse, all_nmse = [], [], []
all_c_means, all_c_stds, all_c_mins, all_c_maxs = [], [], [], []
all_o_means, all_o_stds, all_o_mins, all_o_maxs = [], [], [], []

for run in range(30):
    np.random.seed(run)
    tf.random.set_seed(run)
    tf.keras.backend.clear_session()
    
    model = build_model(input_dim=1, units=8, batch_size=1)
    rnn_layer = model.layers[1]
    
    for epoch in range(100):
        model.fit(X_train, y_train, epochs=1, batch_size=1, verbose=0, shuffle=False)
        rnn_layer.reset_states()

    # Warmup
    for i in range(len(X_train)): model.predict(X_train[i:i+1], batch_size=1, verbose=0)

    predictions, c_gates, o_gates = [], [], []
    for i in range(len(X_test)):
        yhat, c_val, o_val = model.predict(X_test[i:i+1], batch_size=1, verbose=0)
        c_gates.append(c_val[0])
        o_gates.append(o_val[0])
        
        row = list(X_test_raw[i]) + [yhat[0, 0]]
        inv = scaler.inverse_transform([row])[0, -1] + raw_values[len(train) + i]
        predictions.append(inv)
        
    c_gates, o_gates = np.array(c_gates), np.array(o_gates)
    all_c_means.append(np.mean(c_gates)); all_c_stds.append(np.std(c_gates))
    all_c_mins.append(np.min(c_gates)); all_c_maxs.append(np.max(c_gates))
    all_o_means.append(np.mean(o_gates)); all_o_stds.append(np.std(o_gates))
    all_o_mins.append(np.min(o_gates)); all_o_maxs.append(np.max(o_gates))

    actual = raw_values[-60:]
    rmse = sqrt(mean_squared_error(actual, predictions))
    all_rmse.append(rmse)
    print(f'Run {run+1} — RMSE: {rmse:.6f} (Gate Mean C: {np.mean(c_gates):.4f})')

print('\n===== GATE STATISTICS =====')
print(f'C-Gate - Mean: {np.mean(all_c_means):.4f}, Min: {np.min(all_c_mins):.4f}, Max: {np.max(all_c_maxs):.4f}')
print(f'O-Gate - Mean: {np.mean(all_o_means):.4f}, Min: {np.min(all_o_mins):.4f}, Max: {np.max(all_o_maxs):.4f}')
print(f'\nOverall RMSE: {np.mean(all_rmse):.6f}')


Run 1 — RMSE: 0.483517 (Gate Mean C: -0.0376)
Run 2 — RMSE: 0.474003 (Gate Mean C: -0.0695)
Run 3 — RMSE: 0.473426 (Gate Mean C: -0.0366)
Run 4 — RMSE: 0.470675 (Gate Mean C: -0.0674)
Run 5 — RMSE: 0.466456 (Gate Mean C: 0.0507)
Run 6 — RMSE: 0.487889 (Gate Mean C: 0.0019)
Run 7 — RMSE: 0.460517 (Gate Mean C: -0.0033)
Run 8 — RMSE: 0.475882 (Gate Mean C: -0.0582)
Run 9 — RMSE: 0.476968 (Gate Mean C: 0.0167)
Run 10 — RMSE: 0.464310 (Gate Mean C: -0.0369)
Run 11 — RMSE: 0.488306 (Gate Mean C: 0.0124)
Run 12 — RMSE: 0.475660 (Gate Mean C: -0.0504)
Run 13 — RMSE: 0.474312 (Gate Mean C: 0.0017)
Run 14 — RMSE: 0.482899 (Gate Mean C: 0.0095)
Run 15 — RMSE: 0.480730 (Gate Mean C: -0.0178)
Run 16 — RMSE: 0.454751 (Gate Mean C: -0.0807)
Run 17 — RMSE: 0.483180 (Gate Mean C: -0.0007)
Run 18 — RMSE: 0.480201 (Gate Mean C: 0.0171)
Run 19 — RMSE: 0.482550 (Gate Mean C: 0.0070)
Run 20 — RMSE: 0.458797 (Gate Mean C: 0.0528)
Run 21 — RMSE: 0.494773 (Gate Mean C: -0.1048)
Run 22 — RMSE: 0.479690 (Gate M